# Build Metadata Validation (Required Keywords)

This notebook validates a selected experiment JSON against the IUC02 schema by checking required keywords.

It reports warnings when required keywords are missing or present but not defined (for example: `null` or empty string).

## 1) Environment and Imports

In [1]:
import json
import os
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, display

print('Python executable:', sys.executable)
print('Python version   :', sys.version.split()[0])
print('Conda env        :', os.environ.get('CONDA_DEFAULT_ENV', '<not set>'))

Python executable: c:\Users\maria\anaconda3\envs\python311\python.exe
Python version   : 3.11.14
Conda env        : python311


## 2) Project Paths and Validation Helpers

In [2]:
project_root = Path.cwd().resolve().parent
schema_default = project_root.parent / 'Data Schema' / '2024-09_Schema_IUC02_v1.1.json'
experiments_default = project_root / 'Data' / 'BAMDataset_Json'
lis_default_folder = project_root / 'Data' / 'BAMDataset'
mapping_default = project_root / 'Metadata' / 'Mappings' / 'BAM2schema.json'

print('Project root       :', project_root)
print('Default schema path:', schema_default)
print('Default data folder:', experiments_default)
print('Default LIS folder :', lis_default_folder)

def list_json_files(folder_path: Path):
    if not folder_path.exists() or not folder_path.is_dir():
        return []
    return sorted([p for p in folder_path.rglob('*.json') if p.is_file()])

def list_lis_files(folder_path: Path):
    if not folder_path.exists() or not folder_path.is_dir():
        return []
    return sorted([p for p in folder_path.rglob('*.LIS') if p.is_file()])

def candidate_data_folders(data_root: Path):
    if not data_root.exists() or not data_root.is_dir():
        return []

    candidates = [data_root]
    candidates.extend([p for p in data_root.iterdir() if p.is_dir()])

    # Keep only folders containing at least one JSON file.
    out = []
    for folder in candidates:
        if list_json_files(folder):
            out.append(folder)
    return sorted(dict.fromkeys(out))

def candidate_lis_folders(data_root: Path):
    if not data_root.exists() or not data_root.is_dir():
        return []

    candidates = [data_root]
    candidates.extend([p for p in data_root.iterdir() if p.is_dir()])

    out = []
    for folder in candidates:
        if list_lis_files(folder):
            out.append(folder)
    return sorted(dict.fromkeys(out))

def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def resolve_ref(schema_root: dict, schema_node: dict):
    ref = schema_node.get('$ref') if isinstance(schema_node, dict) else None
    if not ref:
        return schema_node
    if not ref.startswith('#/'):
        return schema_node

    target = schema_root
    for part in ref[2:].split('/'):
        target = target.get(part, {}) if isinstance(target, dict) else {}
    return target if isinstance(target, dict) else schema_node

def collect_required_paths(schema_root: dict, schema_node: dict, base_path=()):
    node = resolve_ref(schema_root, schema_node)
    paths = []

    required = node.get('required', []) if isinstance(node, dict) else []
    if isinstance(required, list):
        for key in required:
            if isinstance(key, str):
                paths.append(base_path + (key,))

    properties = node.get('properties', {}) if isinstance(node, dict) else {}
    if isinstance(properties, dict):
        for key, child in properties.items():
            if isinstance(child, dict):
                paths.extend(collect_required_paths(schema_root, child, base_path + (key,)))

    items = node.get('items') if isinstance(node, dict) else None
    if isinstance(items, dict):
        paths.extend(collect_required_paths(schema_root, items, base_path + ('*',)))

    for combiner in ('allOf', 'anyOf', 'oneOf'):
        members = node.get(combiner, []) if isinstance(node, dict) else []
        if isinstance(members, list):
            for member in members:
                if isinstance(member, dict):
                    paths.extend(collect_required_paths(schema_root, member, base_path))

    return paths

def is_defined(value):
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip() != ''
    if isinstance(value, (list, tuple, set, dict)):
        return len(value) > 0
    return True

def check_path_defined(data_node, path_tuple):
    if not path_tuple:
        return True, data_node, None

    head, *tail = path_tuple
    if head == '*':
        if not isinstance(data_node, list):
            return False, None, 'expected array for wildcard segment'
        if not data_node:
            return False, None, 'array is empty'

        for idx, item in enumerate(data_node):
            ok, _, reason = check_path_defined(item, tuple(tail))
            if not ok:
                return False, None, f'array item {idx}: {reason}'
        return True, data_node, None

    if not isinstance(data_node, dict):
        return False, None, 'parent is not an object'
    if head not in data_node:
        return False, None, 'missing key'

    value = data_node[head]
    if not tail:
        return is_defined(value), value, 'not defined' if not is_defined(value) else None

    return check_path_defined(value, tuple(tail))

def normalize_experiment_data(schema_doc: dict, data_doc: dict):
    schema_properties = schema_doc.get('properties', {}) if isinstance(schema_doc, dict) else {}
    schema_target = schema_doc
    data_target = data_doc

    # Many translated files place the schema object under mappedMeasurementData.MeasurementData
    if isinstance(data_doc, dict) and 'mappedMeasurementData' in data_doc:
        mapped = data_doc.get('mappedMeasurementData', {})
        if isinstance(mapped, dict) and 'MeasurementData' in mapped:
            data_target = {'MeasurementData': mapped['MeasurementData']}

    # If data starts directly at MeasurementData payload, wrap it for schema alignment.
    if isinstance(data_target, dict) and 'MeasurementData' not in data_target and 'MeasurementData' in schema_properties:
        if 'additionalMetadata' in data_target or 'primaryData' in data_target or 'secondaryData' in data_target:
            data_target = {'MeasurementData': data_target}

    return schema_target, data_target

def validate_required_keywords(schema_doc: dict, experiment_doc: dict):
    schema_target, data_target = normalize_experiment_data(schema_doc, experiment_doc)
    req_paths = collect_required_paths(schema_target, schema_target)

    # Keep first occurrence order while deduplicating.
    req_paths = list(dict.fromkeys(req_paths))

    warnings = []
    for req in req_paths:
        ok, _, reason = check_path_defined(data_target, req)
        if not ok:
            warnings.append({
                'path': '.'.join(req),
                'reason': reason
            })

    return req_paths, warnings

def convert_lis_to_json(lis_path: Path, output_json_path: Path, mapping_path: Path = mapping_default):
    import importlib

    if not lis_path.exists():
        raise FileNotFoundError(f'LIS file not found: {lis_path}')
    if not mapping_path.exists():
        raise FileNotFoundError(f'Mapping file not found: {mapping_path}')

    output_json_path.parent.mkdir(parents=True, exist_ok=True)

    # Ensure local dependency packages are importable in notebook context.
    lis_pkg_root = project_root / 'dependencies' / 'LISParser'
    mappings_pkg_root = project_root / 'dependencies' / 'Mappingsreader'

    for p in [lis_pkg_root, mappings_pkg_root]:
        p_txt = str(p)
        if p.exists() and p_txt not in sys.path:
            sys.path.insert(0, p_txt)

    try:
        parser_mod = importlib.import_module('LISParser.LisParse')
        map_mod = importlib.import_module('mappingsreader.mapreader')
    except Exception as exc:
        raise ImportError(
            'Could not import LIS conversion modules. Ensure Parsing dependencies are installed and paths are available.'
        ) from exc

    Parser = getattr(parser_mod, 'Parser')
    translate_bam = getattr(map_mod, 'translate_bam')

    mapping_document = load_json(mapping_path)
    lis_parser = Parser(str(lis_path))
    lis_dict = lis_parser.parse_lis()

    metadata = lis_dict.get('metadata', {}) if isinstance(lis_dict, dict) else {}
    translated = translate_bam(metadata, mapping_document)

    with output_json_path.open('w', encoding='utf-8') as f:
        json.dump(translated, f, indent=4, ensure_ascii=False)

    return output_json_path

def tree_html_from_schema(schema_root: dict, schema_node: dict, data_node, req_paths, path=()):
    from html import escape

    node = resolve_ref(schema_root, schema_node) if isinstance(schema_node, dict) else {}
    props = node.get('properties', {}) if isinstance(node, dict) else {}

    if not isinstance(props, dict) or not props:
        if isinstance(data_node, dict):
            return ""
        if isinstance(data_node, list):
            return escape(json.dumps(data_node, ensure_ascii=False))
        if data_node is None:
            return "<span style='color:#a00;font-weight:600;'>(empty)</span>"
        if isinstance(data_node, str) and data_node.strip() == '':
            return "<span style='color:#a00;font-weight:600;'>(empty)</span>"
        return escape(str(data_node))

    html_parts = ["<ul style='list-style-type:none;padding-left:18px;margin:4px 0;'>"]

    for key, child_schema in props.items():
        child_path = path + (key,)
        required_here = key in set(node.get('required', [])) if isinstance(node, dict) else False

        present = isinstance(data_node, dict) and key in data_node
        value = data_node.get(key) if isinstance(data_node, dict) and key in data_node else None

        missing_or_empty = required_here and (not present or not is_defined(value))
        key_style = "color:#a00;font-weight:700;" if missing_or_empty else "color:#1f2937;font-weight:600;"
        req_tag = " <span style='color:#a00;'>(required, missing)</span>" if missing_or_empty else (" <span style='color:#0a7a2a;'>(required)</span>" if required_here else "")

        child_node = resolve_ref(schema_root, child_schema) if isinstance(child_schema, dict) else {}
        child_props = child_node.get('properties', {}) if isinstance(child_node, dict) else {}
        is_branch = isinstance(child_props, dict) and len(child_props) > 0

        if is_branch:
            html_parts.append(
                "<li>"
                f"<details open><summary><span style='{key_style}'>{escape(key)}</span>{req_tag}</summary>"
                f"{tree_html_from_schema(schema_root, child_schema, value if isinstance(value, (dict, list)) else {}, req_paths, child_path)}"
                "</details>"
                "</li>"
            )
        else:
            if not present:
                rendered_val = "<span style='color:#999;'>(not provided)</span>"
            elif value is None or (isinstance(value, str) and value.strip() == ''):
                rendered_val = "<span style='color:#a00;font-weight:600;'>(empty)</span>"
            elif isinstance(value, (dict, list)):
                rendered_val = escape(json.dumps(value, ensure_ascii=False))
            else:
                rendered_val = escape(str(value))

            html_parts.append(
                "<li>"
                f"<span style='{key_style}'>{escape(key)}</span>{req_tag}: "
                f"<span>{rendered_val}</span>"
                "</li>"
            )

    html_parts.append("</ul>")
    return ''.join(html_parts)

Project root       : C:\Users\maria\Desktop\IUC02\iuc02\Parsing
Default schema path: C:\Users\maria\Desktop\IUC02\iuc02\Data Schema\2024-09_Schema_IUC02_v1.1.json
Default data folder: C:\Users\maria\Desktop\IUC02\iuc02\Parsing\Data\BAMDataset_Json


## 3) Select JSON for Validation or Convert LIS to JSON

In [ ]:
schema_path_text = widgets.Text(
    value=str(schema_default),
    description='Schema:',
    layout=widgets.Layout(width='98%')
)

data_root = project_root / 'Data'
folder_dropdown = widgets.Dropdown(
    options=[],
    description='JSON Folder:',
    layout=widgets.Layout(width='98%')
)

refresh_folders_button = widgets.Button(description='Refresh JSON folders', button_style='info')
refresh_files_button = widgets.Button(description='Load JSON files')

experiment_dropdown = widgets.Dropdown(
    options=[],
    description='Experiment JSON:',
    layout=widgets.Layout(width='98%')
)

lis_folder_dropdown = widgets.Dropdown(
    options=[],
    description='LIS Folder:',
    layout=widgets.Layout(width='98%')
)
refresh_lis_folders_button = widgets.Button(description='Refresh LIS folders', button_style='info')
refresh_lis_files_button = widgets.Button(description='Load LIS files')

lis_file_dropdown = widgets.Dropdown(
    options=[],
    description='LIS File:',
    layout=widgets.Layout(width='98%')
)

converted_json_name_text = widgets.Text(
    value='selected_from_lis_translated.json',
    description='Output JSON:',
    layout=widgets.Layout(width='98%')
)
convert_lis_button = widgets.Button(description='Convert LIS to JSON', button_style='warning')

status_html = widgets.HTML()
selection_html = widgets.HTML()

def refresh_folder_dropdown(*_):
    folders = candidate_data_folders(data_root)

    if not folders:
        folder_dropdown.options = [('(no folders with JSON found)', '')]
        folder_dropdown.value = ''
        experiment_dropdown.options = [('(no JSON files found)', '')]
        experiment_dropdown.value = ''
        status_html.value = f"<span style='color:#b00020;'>No data folders with JSON files found in: {data_root}</span>"
        update_selection()
        return

    folder_dropdown.options = [(str(p.relative_to(project_root)), str(p)) for p in folders]

    preferred = str(experiments_default) if experiments_default in folders else str(folders[0])
    folder_dropdown.value = preferred
    refresh_experiment_dropdown()

def refresh_experiment_dropdown(*_):
    folder_value = folder_dropdown.value
    if not folder_value:
        experiment_dropdown.options = [('(select a folder first)', '')]
        experiment_dropdown.value = ''
        update_selection()
        return

    folder = Path(folder_value)
    files = list_json_files(folder)

    if not files:
        experiment_dropdown.options = [('(no JSON files found)', '')]
        experiment_dropdown.value = ''
        status_html.value = f"<span style='color:#b00020;'>No JSON files found in: {folder}</span>"
        update_selection()
        return

    options = [(str(p.relative_to(folder)), str(p)) for p in files]
    experiment_dropdown.options = options
    experiment_dropdown.value = str(files[0])
    status_html.value = f"<span style='color:#0a7a2a;'>Loaded {len(files)} JSON files from: {folder}</span>"
    update_selection()

def refresh_lis_folder_dropdown(*_):
    folders = candidate_lis_folders(data_root)

    if not folders:
        lis_folder_dropdown.options = [('(no folders with LIS found)', '')]
        lis_folder_dropdown.value = ''
        lis_file_dropdown.options = [('(no LIS files found)', '')]
        lis_file_dropdown.value = ''
        update_selection()
        return

    lis_folder_dropdown.options = [(str(p.relative_to(project_root)), str(p)) for p in folders]
    preferred = str(lis_default_folder) if lis_default_folder in folders else str(folders[0])
    lis_folder_dropdown.value = preferred
    refresh_lis_file_dropdown()

def refresh_lis_file_dropdown(*_):
    folder_value = lis_folder_dropdown.value
    if not folder_value:
        lis_file_dropdown.options = [('(select LIS folder first)', '')]
        lis_file_dropdown.value = ''
        update_selection()
        return

    folder = Path(folder_value)
    files = list_lis_files(folder)

    if not files:
        lis_file_dropdown.options = [('(no LIS files found)', '')]
        lis_file_dropdown.value = ''
        update_selection()
        return

    options = [(str(p.relative_to(folder)), str(p)) for p in files]
    lis_file_dropdown.options = options
    lis_file_dropdown.value = str(files[0])
    update_selection()

def convert_selected_lis(_):
    lis_value = lis_file_dropdown.value
    json_folder_value = folder_dropdown.value
    out_name = converted_json_name_text.value.strip()

    if not lis_value:
        status_html.value = "<span style='color:#b00020;'>Select a LIS file first.</span>"
        return
    if not json_folder_value:
        status_html.value = "<span style='color:#b00020;'>Select a JSON output folder first.</span>"
        return
    if not out_name:
        status_html.value = "<span style='color:#b00020;'>Provide an output JSON filename.</span>"
        return

    out_name = out_name if out_name.lower().endswith('.json') else f'{out_name}.json'
    output_json_path = Path(json_folder_value) / out_name

    try:
        converted_path = convert_lis_to_json(Path(lis_value), output_json_path)
        refresh_experiment_dropdown()

        # Select the converted file automatically for validation.
        for _, file_value in experiment_dropdown.options:
            if file_value == str(converted_path):
                experiment_dropdown.value = file_value
                break

        status_html.value = (
            f"<span style='color:#0a7a2a;'>Converted LIS to JSON successfully:</span> "
            f"<code>{converted_path}</code>"
        )
        update_selection()
    except Exception as exc:
        status_html.value = f"<span style='color:#b00020;'>LIS conversion failed: {exc}</span>"

def update_selection(*_):
    schema_txt = schema_path_text.value.strip()
    folder_txt = folder_dropdown.value or '(none selected)'
    exp_txt = experiment_dropdown.value or '(none selected)'
    lis_folder_txt = lis_folder_dropdown.value or '(none selected)'
    lis_txt = lis_file_dropdown.value or '(none selected)'

    selection_html.value = (
        f'<b>Schema:</b> {schema_txt}<br>'
        f'<b>JSON folder:</b> {folder_txt}<br>'
        f'<b>Experiment JSON:</b> {exp_txt}<br>'
        f'<b>LIS folder:</b> {lis_folder_txt}<br>'
        f'<b>LIS file:</b> {lis_txt}'
    )

refresh_folders_button.on_click(refresh_folder_dropdown)
refresh_files_button.on_click(refresh_experiment_dropdown)
refresh_lis_folders_button.on_click(refresh_lis_folder_dropdown)
refresh_lis_files_button.on_click(refresh_lis_file_dropdown)
convert_lis_button.on_click(convert_selected_lis)

schema_path_text.observe(update_selection, names='value')
folder_dropdown.observe(refresh_experiment_dropdown, names='value')
experiment_dropdown.observe(update_selection, names='value')
lis_folder_dropdown.observe(refresh_lis_file_dropdown, names='value')
lis_file_dropdown.observe(update_selection, names='value')
converted_json_name_text.observe(update_selection, names='value')

refresh_folder_dropdown()
refresh_lis_folder_dropdown()
display(
    widgets.VBox([
        schema_path_text,
        widgets.HTML('<hr><b>Choose JSON for Validation</b>'),
        widgets.HBox([refresh_folders_button, refresh_files_button]),
        folder_dropdown,
        experiment_dropdown,
        widgets.HTML('<hr><b>Or Convert LIS to JSON</b>'),
        widgets.HBox([refresh_lis_folders_button, refresh_lis_files_button]),
        lis_folder_dropdown,
        lis_file_dropdown,
        converted_json_name_text,
        convert_lis_button,
        status_html,
        selection_html
    ])
)

## 4) Run Validation and Show Warnings

In [4]:
schema_path = Path(schema_path_text.value.strip())
experiment_path = Path(experiment_dropdown.value) if experiment_dropdown.value else None

if not schema_path.exists():
    raise FileNotFoundError(f'Schema file not found: {schema_path}')
if experiment_path is None or not experiment_path.exists():
    raise FileNotFoundError('Please select a valid experiment JSON file.')

schema_doc = load_json(schema_path)
experiment_doc = load_json(experiment_path)
required_paths, warnings = validate_required_keywords(schema_doc, experiment_doc)

print(f'Experiment file checked: {experiment_path}')
print(f'Total required keywords declared in schema: {len(required_paths)}')

if warnings:
    print(f'Warnings: {len(warnings)} required keyword(s) are missing or not defined.')
    warning_rows = ''.join([
        f"<tr><td style='padding:4px;border:1px solid #ddd;'>{w['path']}</td><td style='padding:4px;border:1px solid #ddd;'>{w['reason']}</td></tr>"
        for w in warnings
    ])
    html = f"""
    <div style='background:#fff4e5;border-left:5px solid #e67e22;padding:12px;margin:10px 0;'>
      <b>Warning:</b> Some required keywords are missing or undefined.
    </div>
    <table style='border-collapse:collapse;width:100%;font-family:Arial,sans-serif;font-size:13px;'>
      <thead>
        <tr style='background:#f7f7f7;'>
          <th style='text-align:left;padding:6px;border:1px solid #ddd;'>Required keyword path</th>
          <th style='text-align:left;padding:6px;border:1px solid #ddd;'>Issue</th>
        </tr>
      </thead>
      <tbody>{warning_rows}</tbody>
    </table>
    """
    display(HTML(html))
else:
    display(HTML("<div style='background:#eafaf1;border-left:5px solid #1e8449;padding:12px;'><b>Success:</b> All required keywords are defined.</div>"))

Experiment file checked: C:\Users\maria\Desktop\IUC02\iuc02\Parsing\Data\BAMDataset_Json\Vh5205_C-95_translated.json
Total required keywords declared in schema: 121
Warnings: 98 required keyword(s) are missing or not defined.


Required keyword path,Issue
MeasurementData.additionalMetadata.testInfo.testJobDetails.dateOfTestEnd,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.testStandardApplied,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.testingStandard,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.typeOfLoading,not defined
MeasurementData.additionalMetadata.testInfo.testParameters.loadControlType,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.testType,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.endOfTestCriterium,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.endOfTestCriteriumValue,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.interruptionCourse,missing key
MeasurementData.additionalMetadata.testInfo.testParameters.preloadTestForce,missing key


## 5) Tree View of Fields and Values

In [5]:
schema_target, data_target = normalize_experiment_data(schema_doc, experiment_doc)
req_paths = collect_required_paths(schema_target, schema_target)

# Deduplicate while preserving order.
req_paths = list(dict.fromkeys(req_paths))

tree_html = tree_html_from_schema(schema_target, schema_target, data_target, req_paths)

legend_html = """
<div style='margin:8px 0 12px 0;font-family:Arial,sans-serif;font-size:13px;'>
  <span style='color:#0a7a2a;font-weight:600;'>(required)</span> = required and defined
  <br>
  <span style='color:#a00;font-weight:700;'>(required, missing)</span> = required but missing/empty
</div>
"""

display(HTML(legend_html + tree_html))

## 6) Full JSON Schema Validation (Types, Enums, Patterns)

In [6]:
try:
    from jsonschema import Draft201909Validator
except ImportError as exc:
    raise ImportError(
        "jsonschema is not installed in this environment. Install it with: pip install jsonschema"
    ) from exc

schema_target, data_target = normalize_experiment_data(schema_doc, experiment_doc)

validator = Draft201909Validator(schema_target)
errors = sorted(validator.iter_errors(data_target), key=lambda e: list(e.path))

if not errors:
    display(HTML("<div style='background:#eafaf1;border-left:5px solid #1e8449;padding:12px;'><b>Schema valid:</b> JSON Schema constraints are satisfied.</div>"))
else:
    print(f"Schema errors found: {len(errors)}")

    max_to_show = 100
    rows = []
    for err in errors[:max_to_show]:
        data_path = '.'.join([str(p) for p in err.path]) if list(err.path) else '<root>'
        schema_path_txt = '/'.join([str(p) for p in err.schema_path])
        message = str(err.message).replace('<', '&lt;').replace('>', '&gt;')

        rows.append(
            f"<tr>"
            f"<td style='padding:4px;border:1px solid #ddd;vertical-align:top;'>{data_path}</td>"
            f"<td style='padding:4px;border:1px solid #ddd;vertical-align:top;'>{message}</td>"
            f"<td style='padding:4px;border:1px solid #ddd;vertical-align:top;'>{schema_path_txt}</td>"
            f"</tr>"
        )

    if len(errors) > max_to_show:
        rows.append(
            f"<tr><td colspan='3' style='padding:6px;border:1px solid #ddd;color:#555;'>"
            f"Showing first {max_to_show} errors out of {len(errors)}."
            f"</td></tr>"
        )

    html = (
        "<div style='background:#fdecea;border-left:5px solid #c0392b;padding:12px;margin:10px 0;'>"
        "<b>Schema validation warning:</b> The experiment JSON does not fully match the schema."
        "</div>"
        "<table style='border-collapse:collapse;width:100%;font-family:Arial,sans-serif;font-size:13px;'>"
        "<thead>"
        "<tr style='background:#f7f7f7;'>"
        "<th style='text-align:left;padding:6px;border:1px solid #ddd;'>Data path</th>"
        "<th style='text-align:left;padding:6px;border:1px solid #ddd;'>Issue</th>"
        "<th style='text-align:left;padding:6px;border:1px solid #ddd;'>Schema path</th>"
        "</tr>"
        "</thead>"
        f"<tbody>{''.join(rows)}</tbody>"
        "</table>"
    )

    display(HTML(html))

Schema errors found: 48


Data path,Issue,Schema path
MeasurementData.additionalMetadata.measuringAndTestEquipment.extensometerSystem,'strainMeasuringDeviceType' is a required property,properties/MeasurementData/properties/additionalMetadata/properties/measuringAndTestEquipment/properties/extensometerSystem/required
MeasurementData.additionalMetadata.measuringAndTestEquipment.extensometerSystem,'nonContactingExtensometerSensorType' is a required property,properties/MeasurementData/properties/additionalMetadata/properties/measuringAndTestEquipment/properties/extensometerSystem/required
MeasurementData.additionalMetadata.measuringAndTestEquipment.extensometerSystem.contactingExtensometerSensorType,None is not of type 'string',properties/MeasurementData/properties/additionalMetadata/properties/measuringAndTestEquipment/properties/extensometerSystem/properties/contactingExtensometerSensorType/type
MeasurementData.additionalMetadata.measuringAndTestEquipment.extensometerSystem.contactingExtensometerSensorType,"None is not one of ['LVDT with extension legs', 'Clip-on extensometer']",properties/MeasurementData/properties/additionalMetadata/properties/measuringAndTestEquipment/properties/extensometerSystem/properties/contactingExtensometerSensorType/enum
MeasurementData.additionalMetadata.testInfo.testJobDetails,'dateOfTestEnd' is a required property,properties/MeasurementData/properties/additionalMetadata/properties/testInfo/properties/testJobDetails/required
MeasurementData.additionalMetadata.testInfo.testJobDetails.testID,520595.0 is not of type 'string',properties/MeasurementData/properties/additionalMetadata/properties/testInfo/properties/testJobDetails/properties/testID/type
MeasurementData.additionalMetadata.testInfo.testParameters,'testStandardApplied' is a required property,properties/MeasurementData/properties/additionalMetadata/properties/testInfo/properties/testParameters/required
MeasurementData.additionalMetadata.testInfo.testParameters,'testingStandard' is a required property,properties/MeasurementData/properties/additionalMetadata/properties/testInfo/properties/testParameters/required
MeasurementData.additionalMetadata.testInfo.testParameters,'loadControlType' is a required property,properties/MeasurementData/properties/additionalMetadata/properties/testInfo/properties/testParameters/required
MeasurementData.additionalMetadata.testInfo.testParameters,'testType' is a required property,properties/MeasurementData/properties/additionalMetadata/properties/testInfo/properties/testParameters/required


## Notes
- Section 3 allows direct JSON selection or LIS-to-JSON conversion.
- Section 4 checks required keyword presence and whether values are defined.
- Section 5 renders a schema-based tree of fields and values and highlights missing required values in red.
- Section 6 performs full JSON Schema validation (for example: `type`, `enum`, and pattern constraints).